https://github.com/openai/whisper

Whisper官方仓库

# 请指导我在colab上部署开源的STT模型，并将本地文件上传到colab，并将录音文件转成文本


好的，这是一个非常棒的实践选择！在 **Google Colab** 上部署开源的 STT 模型（特别是 **Whisper**）是最佳实践，因为 Colab 提供了免费的 GPU 资源，能让 Whisper 的识别速度飞快。

我们将采用 **OpenAI 的 Whisper 模型**，因为它在准确率和多语言支持方面表现出色。

请按照以下 **分步指导** 在 Colab 上完成部署、文件上传和语音转文本的全过程。

---

## 🚀 Colab 实战：Whisper 语音转文本

### 第一步：设置 Colab 环境和安装依赖

首先，确保您的 Colab 笔记本已连接到高性能运行时。

1.  **检查并切换运行时类型（GPU 加速）：**
    *   点击 Colab 菜单栏的 **“运行时 (Runtime)”** -> **“更改运行时类型 (Change runtime type)”**。
    *   在 **“硬件加速器 (Hardware accelerator)”** 下拉菜单中，选择 **“T4 GPU”** 或 **“A100 GPU”**。
    *   点击 **“保存”**。

2.  **安装所需库：**
    在 Colab 的第一个代码单元格中，运行以下命令来安装 Whisper 及其依赖。

    ```python
    # 安装 Whisper
    !pip install -q git+https://github.com/openai/whisper.git

    # 安装 PyTorch 的音频处理库 torchaudio 和音频编解码器
    # -q 表示静默安装，让输出更简洁
    !pip install -q torchaudio
    
    # 安装 ffmpeg 依赖库，Whisper 用它来处理音频文件
    !apt update && apt install -y ffmpeg
    
    print("✅ Whisper 及其依赖安装完成！")
    ```

### 第二步：上传本地 WAV 录音文件

Colab 笔记本在每次会话关闭后会清除文件，所以我们需要将本地文件上传到当前的 Colab 环境中。

在第二个代码单元格中，运行以下 Python 代码来启动上传界面：

```python
from google.colab import files
import os

# 定义录音文件需要保存的名称
# 请注意，您上传的文件名必须与这个变量匹配，或者在后续代码中修改
AUDIO_FILENAME = "my_recording.wav"

print(f"请点击 '选择文件' 按钮，上传您的 WAV 录音文件。")

# 启动文件上传界面
uploaded = files.upload()

# 检查上传的文件
if uploaded:
    # Colab 上传的文件会以字典形式存储，键是文件名，值是二进制数据
    # 我们只关心第一个文件（假设您只上传了一个）
    uploaded_filename = list(uploaded.keys())[0]
    
    # 将上传的文件重命名为我们预设的名称，方便后续代码引用
    if uploaded_filename != AUDIO_FILENAME:
        os.rename(uploaded_filename, AUDIO_FILENAME)
        print(f"✅ 文件 '{uploaded_filename}' 已成功上传并重命名为 '{AUDIO_FILENAME}'。")
    else:
        print(f"✅ 文件 '{AUDIO_FILENAME}' 已成功上传。")
    
    # 验证文件是否存在
    if os.path.exists(AUDIO_FILENAME):
        print(f"文件大小: {os.path.getsize(AUDIO_FILENAME) / (1024*1024):.2f} MB")
    else:
        print("❌ 文件上传失败，请检查。")
else:
    print("⚠️ 未选择任何文件进行上传。")
```

> **操作步骤：** 运行上面的代码单元格后，会弹出一个 **“选择文件”** 按钮。点击它，从您的本地电脑中选择您的 `wav` 录音文件。

### 第三步：加载 Whisper 模型并进行识别

现在，我们来编写核心的语音转文本代码。

在第三个代码单元格中，运行以下代码：

```python
import whisper
import time

# --- 配置参数 ---
# 待识别的文件名，必须与第二步上传和重命名的文件一致
INPUT_FILE = "my_recording.wav"

# 选择 Whisper 模型大小。
# 'base'：速度快，占用内存低，适合初学者和较短音频。
# 'medium'：准确率更高，速度适中，推荐用于中文识别。
# 'large'：最高准确率，速度最慢。
MODEL_SIZE = "medium"

# 识别的语言代码。指定语言可以显著提高识别准确率和速度。
# 中文普通话：zh
# 英文：en
LANGUAGE = "zh"

print(f"--- 语音识别程序启动 ---")
print(f"模型大小: {MODEL_SIZE}, 目标语言: {LANGUAGE}")

# 1. 加载 Whisper 模型
# to() 函数将模型加载到 GPU 上 (cuda)，如果无法使用 GPU，它会自动退回到 CPU。
try:
    start_load_time = time.time()
    # whisper.load_model 会自动下载模型文件（如果本地没有）
    model = whisper.load_model(MODEL_SIZE).to("cuda")
    load_time = time.time() - start_load_time
    print(f"✅ 模型加载完成 (耗时: {load_time:.2f} 秒)")
except Exception as e:
    print(f"❌ 模型加载失败，可能没有GPU或网络问题: {e}")
    # 尝试加载到CPU
    model = whisper.load_model(MODEL_SIZE)
    print("✅ 已退回到 CPU 模式加载模型。")


# 2. 调用模型进行识别
if 'model' in locals():
    print(f"🚀 正在识别文件: {INPUT_FILE}...")
    start_transcribe_time = time.time()
    
    # model.transcribe 是 Whisper 的核心函数
    result = model.transcribe(
        audio=INPUT_FILE,
        language=LANGUAGE,
        # task="transcribe" 是默认值，表示转文本。也可以设置为 "translate" 进行翻译
        task="transcribe"
    )
    
    transcribe_time = time.time() - start_transcribe_time
    
    # 3. 输出结果
    print("\n" + "="*50)
    print(f"🎉 识别完成！总耗时: {transcribe_time:.2f} 秒")
    print("="*50)
    
    # result 是一个字典，'text' 键包含完整的识别文本
    print("📝 最终识别文本：")
    print(result["text"])
    print("="*50)

```

### 总结和清理

现在，您已经成功地在 Colab 上使用 Whisper 模型将本地 WAV 文件转换为了文本。

**核心原理：**

1.  **`!pip install` / `!apt install`：** 在 Colab 虚拟机中安装了 Python 库和系统工具。
2.  **`files.upload()`：** 利用 Colab 的内置功能，将您的本地文件传输到云端虚拟机。
3.  **`whisper.load_model()`：** 从 OpenAI 的服务器下载预训练的模型参数，并将其加载到 Colab 提供的 GPU 内存中。
4.  **`model.transcribe()`：** 模型对音频进行切片、特征提取、然后使用强大的 Transformer 神经网络将声音特征解码为文本。GPU 的存在使得这一复杂计算过程变得非常快。

您现在可以尝试上传不同的录音文件，或者调整 `MODEL_SIZE` 来观察准确率和速度的变化！

# 第一步：设置 Colab 环境和安装依赖

In [2]:
# 安装 Whisper
!pip install -q git+https://github.com/openai/whisper.git

# 安装 PyTorch 的音频处理库 torchaudio 和音频编解码器
# -q 表示静默安装，让输出更简洁
!pip install -q torchaudio

# 安装 ffmpeg 依赖库，Whisper 用它来处理音频文件
!apt update && apt install -y ffmpeg

print("✅ Whisper 及其依赖安装完成！")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,817 kB]
Get:12 http://security.ubuntu.com/ubuntu jamm

# 第二步：上传本地 WAV 录音文件

In [5]:
from google.colab import files
import os

# 定义录音文件需要保存的名称
# 请注意，您上传的文件名必须与这个变量匹配，或者在后续代码中修改
AUDIO_FILENAME = "my_recording.wav"

print(f"请点击 '选择文件' 按钮，上传您的 WAV 录音文件。")

# 启动文件上传界面
uploaded = files.upload()

# 检查上传的文件
if uploaded:
    # Colab 上传的文件会以字典形式存储，键是文件名，值是二进制数据
    # 我们只关心第一个文件（假设您只上传了一个）
    uploaded_filename = list(uploaded.keys())[0]

    # 将上传的文件重命名为我们预设的名称，方便后续代码引用
    if uploaded_filename != AUDIO_FILENAME:
        os.rename(uploaded_filename, AUDIO_FILENAME)
        print(f"✅ 文件 '{uploaded_filename}' 已成功上传并重命名为 '{AUDIO_FILENAME}'。")
    else:
        print(f"✅ 文件 '{AUDIO_FILENAME}' 已成功上传。")

    # 验证文件是否存在
    if os.path.exists(AUDIO_FILENAME):
        print(f"文件大小: {os.path.getsize(AUDIO_FILENAME) / (1024*1024):.2f} MB")
    else:
        print("❌ 文件上传失败，请检查。")
else:
    print("⚠️ 未选择任何文件进行上传。")


请点击 '选择文件' 按钮，上传您的 WAV 录音文件。


Saving 洛天依1.wav to 洛天依1.wav
✅ 文件 '洛天依1.wav' 已成功上传并重命名为 'my_recording.wav'。
文件大小: 4.53 MB


# 第三步：加载 Whisper 模型并进行识别

In [6]:
import whisper
import time

# --- 配置参数 ---
# 待识别的文件名，必须与第二步上传和重命名的文件一致
INPUT_FILE = "my_recording.wav"

# 选择 Whisper 模型大小。
# 'base'：速度快，占用内存低，适合初学者和较短音频。
# 'medium'：准确率更高，速度适中，推荐用于中文识别。
# 'large'：最高准确率，速度最慢。
MODEL_SIZE = "medium"

# 识别的语言代码。指定语言可以显著提高识别准确率和速度。
# 中文普通话：zh
# 英文：en
LANGUAGE = "zh"

print(f"--- 语音识别程序启动 ---")
print(f"模型大小: {MODEL_SIZE}, 目标语言: {LANGUAGE}")

# 1. 加载 Whisper 模型
# to() 函数将模型加载到 GPU 上 (cuda)，如果无法使用 GPU，它会自动退回到 CPU。
try:
    start_load_time = time.time()
    # whisper.load_model 会自动下载模型文件（如果本地没有）
    model = whisper.load_model(MODEL_SIZE).to("cuda")
    load_time = time.time() - start_load_time
    print(f"✅ 模型加载完成 (耗时: {load_time:.2f} 秒)")
except Exception as e:
    print(f"❌ 模型加载失败，可能没有GPU或网络问题: {e}")
    # 尝试加载到CPU
    model = whisper.load_model(MODEL_SIZE)
    print("✅ 已退回到 CPU 模式加载模型。")


# 2. 调用模型进行识别
if 'model' in locals():
    print(f"🚀 正在识别文件: {INPUT_FILE}...")
    start_transcribe_time = time.time()

    # model.transcribe 是 Whisper 的核心函数
    result = model.transcribe(
        audio=INPUT_FILE,
        language=LANGUAGE,
        # task="transcribe" 是默认值，表示转文本。也可以设置为 "translate" 进行翻译
        task="transcribe"
    )

    transcribe_time = time.time() - start_transcribe_time

    # 3. 输出结果
    print("\n" + "="*50)
    print(f"🎉 识别完成！总耗时: {transcribe_time:.2f} 秒")
    print("="*50)

    # result 是一个字典，'text' 键包含完整的识别文本
    print("📝 最终识别文本：")
    print(result["text"])
    print("="*50)



--- 语音识别程序启动 ---
模型大小: medium, 目标语言: zh


100%|█████████████████████████████████████| 1.42G/1.42G [00:16<00:00, 92.7MiB/s]


✅ 模型加载完成 (耗时: 30.81 秒)
🚀 正在识别文件: my_recording.wav...

🎉 识别完成！总耗时: 4.23 秒
📝 最终识别文本：
哎呀,你這樣撒嬌,我都不好意思了嘛,輕輕笑著不過,既然你這麼可愛,那我就抱抱你吧。張開雙臂,溫柔的抱住你怎麼樣,感覺溫暖嗎?


# review和reflection执行结果

太棒了！您执行得非常完美，每一步都按计划顺利完成，并且成功获得了高质量的文本识别结果。

让我们来回顾一下您的执行过程，并解读输出信息：

### 您的执行结果回顾

| 步骤 | 输出信息 | 结果分析 |
| :--- | :--- | :--- |
| **步骤一：环境配置** | `...Building wheel for openai-whisper...` | Whisper 库安装成功。 |
| | `ffmpeg is already the newest version...` | 音频处理工具 `ffmpeg` 在 Colab 中已预装，可以直接使用。 |
| | `✅ Whisper 及其依赖安装完成！` | 确认环境准备就绪。 |
| **步骤二：文件上传** | `洛天依1.wav (audio/wav) - 4747820 bytes...` | 您成功上传了名为 `洛天依1.wav` 的文件。 |
| | `✅ 文件 '洛天依1.wav' 已成功上传并重命名为 'my_recording.wav'。` | 文件已根据脚本要求重命名，确保了后续代码能够正确引用。文件大小约为 4.53MB。 |
| **步骤三：模型识别** | `模型大小: medium, 目标语言: zh` | 确认使用了准确率较高的 `medium` 模型和中文识别。 |
| | `100%|████...| 1.42G/1.42G [00:16<00:00, 92.7MiB/s]` | 这是 Colab 正在下载 `medium` 模型（大小约 1.42 GB）的进度条。 |
| | `✅ 模型加载完成 (耗时: 30.81 秒)` | 模型已成功加载到 Colab 的 GPU 上。 |
| | `🎉 识别完成！总耗时: 4.23 秒` | **关键指标！** 4.53MB 的音频文件，识别只用了 4.23 秒，这充分体现了 Colab GPU 加速的强大优势。如果使用纯 CPU 识别，这个时间可能会长达几分钟。 |
| **最终结果** | `📝 最终识别文本：哎呀,你這樣撒嬌,我都不好意思了嘛,輕輕笑著不過,既然你這麼可愛,那我就抱抱你吧。張開雙臂,溫柔的抱住你怎麼樣,感覺溫暖嗎?` | 语音成功转换为文本，识别结果非常自然和连贯。 |

### 结论

您成功地在 Colab 上部署了 Whisper 开源 STT 模型，并完成了本地 WAV 文件的上传和转换。

**识别文本：**
> 哎呀,你這樣撒嬌,我都不好意思了嘛,輕輕笑著不過,既然你這麼可愛,那我就抱抱你吧。張開雙臂,溫柔的抱住你怎麼樣,感覺溫暖嗎?

这个结果显示了 Whisper 模型处理中文语音时的超高准确性和对口语化表达的良好理解。

如果您有任何其他关于模型调整、处理批量文件、或者想尝试其他 STT 模型（如本地运行的 Vosk 或 Kaldi）的技术问题，随时可以问我！